# 01 · Constitution du mini-jeu de données
Objectif : 30 à 60 photos de portes marocaines, licence vérifiée, légendées.
Les fichiers `data/sources.csv` et `data/captions.csv` sont légers : ils vont dans Git.

### Installation (à lancer à chaque nouvelle session Colab)
Exécution > Modifier le type d'exécution > **GPU T4**.

In [ ]:
%cd /content
!git clone https://github.com/oussioumaima7-star/lora-portes-marocaines.git 2>/dev/null || git -C lora-portes-marocaines pull
%cd /content/lora-portes-marocaines
!pip install -q -r requirements.txt

Les LoRA et les images brutes sont perdues à la fin de la session : on les stocke sur Google Drive et on crée des liens symboliques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lora_projet/{loras,raw,processed}
!rm -rf results/loras data/raw data/processed
!ln -s /content/drive/MyDrive/lora_projet/loras results/loras
!ln -s /content/drive/MyDrive/lora_projet/raw data/raw
!ln -s /content/drive/MyDrive/lora_projet/processed data/processed
# sources.csv et captions.csv sont aussi gardés sur Drive (sinon perdus à la déconnexion)
!D=/content/drive/MyDrive/lora_projet; for f in sources.csv captions.csv; do [ -f $D/$f ] || cp data/$f $D/$f 2>/dev/null; rm -f data/$f; ln -s $D/$f data/$f; done

## 1. Télécharger des candidates depuis Wikimedia Commons

In [ ]:
!python data/download_commons.py --category "Doors in Morocco" --depth 1 --max 150 --thumb-width 960

### Si `sources.csv` a été perdu
Ne PAS relancer le téléchargement (il re-téléchargerait les images supprimées au tri). Reconstruire le fichier à partir des images présentes sur Drive :

In [ ]:
!python data/rebuild_sources.py --category "Doors in Morocco" --depth 1

## 2. Tri manuel (étape la plus importante)
Supprimer de `data/raw/` : plusieurs portes sur la photo, personnes au premier plan, filigranes, porte coupée, flou, vue de trop loin. Garder **un seul concept**.
Puis retirer les lignes correspondantes de `data/sources.csv` (cellule suivante).

In [ ]:
!ls data/raw | wc -l
import pandas as pd, os
src = pd.read_csv('data/sources.csv')
src = src[src.file_name.apply(lambda f: os.path.exists(f'data/raw/{f}'))]
src.to_csv('data/sources.csv', index=False)
print(len(src), 'images conservées'); src.license.value_counts()

## 3. Préparation 512×512, doublons, découpage train / heldout

In [ ]:
!python data/prepare.py --heldout 8 --mode crop
!cat data/prepare_report.txt

In [ ]:
from PIL import Image
import glob, matplotlib.pyplot as plt
fs = sorted(glob.glob('data/processed/train/*.png'))[:16]
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for a, f in zip(ax.flat, fs): a.imshow(Image.open(f)); a.axis('off')

## 4. Légendes : pré-remplissage BLIP puis **correction manuelle**
Format conseillé : `a photo of a mrcdoor, <couleur>, <matière>, <décor autour>`.
Décrire ce qui VARIE ; le mot `mrcdoor` absorbe ce qui est commun.

In [ ]:
!python -m src.auto_caption
pd.read_csv('data/captions.csv')

Ouvrir `data/captions.csv`, corriger chaque ligne, puis commit + push.